In [ ]:
""" 
Calibration curve for my brain-genotype scores 
Simple calibration plots for multi-class genotype probs
-Per class, top predicted class, calculated score dosage 
Run in a Jupyter cell. Requires: pandas, numpy, matplotlib, sklearn
"""


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss

# ----------------- USER: set file path -----------------
input_path = '/path/to/data/example_scores.csv'
# If you leave input_path = None, demo data will be used
# -----------------------------------------------------
df = pd.read_csv(input_path)
    
# ---------- Adjust these names if your file uses different headers ----------
prob0_col = "Prob_Class_0"
prob1_col = "Prob_Class_1"
prob2_col = "Prob_Class_2"
true_col  = "True_Label"
# -------------------------------------------------------------------------

# quick checks
for c in (prob0_col, prob1_col, prob2_col, true_col):
    if c not in df.columns:
        raise ValueError(f"Column {c} not found in file. Found columns: {df.columns.tolist()}")

labels = df[true_col].values
if not np.all(np.isin(labels, [0,1,2])):
    raise ValueError(f"True labels must be in {{0,1,2}}. Found: {np.unique(labels)}")

# ensure numeric
df[[prob0_col, prob1_col, prob2_col]] = df[[prob0_col, prob1_col, prob2_col]].astype(float)
df[true_col] = df[true_col].astype(int)

probs = df[[prob0_col, prob1_col, prob2_col]].values
n = len(df)
print(f"Loaded {n} rows.")

# Multiclass Brier
# multiclass Brier = mean over samples of sum_c (p_ic - y_ic)^2
y_onehot = np.zeros_like(probs)
y_onehot[np.arange(n), df[true_col].values] = 1
brier_multi = np.mean(np.sum((probs - y_onehot)**2, axis=1))
print(f"Multiclass Brier: {brier_multi:.6f}")

# Per-class Brier and simple ECE (equal-width bins)
def ece_bin(p, y, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins+1)
    idx = np.digitize(p, bins) - 1
    ece = 0.0
    for i in range(n_bins):
        mask = idx == i
        if mask.sum()==0:
            continue
        ece += (mask.sum()/len(p)) * abs(p[mask].mean() - y[mask].mean())
    return ece

for i, col in enumerate((prob0_col, prob1_col, prob2_col)):
    p = df[col].values
    y = (df[true_col].values == i).astype(int)
    brier = brier_score_loss(y, p)
    ece = ece_bin(p, y, n_bins=10)
    print(f"Class {i}: Brier={brier:.6f}, ECE(10bins)={ece:.6f}")

# Top-class (max prob) metrics
pred_class = probs.argmax(axis=1)
p_max = probs.max(axis=1)
y_top = (pred_class == df[true_col].values).astype(int)
brier_top = brier_score_loss(y_top, p_max)
ece_top = ece_bin(p_max, y_top, n_bins=10)
print(f"Top-class: Brier={brier_top:.6f}, ECE(10bins)={ece_top:.6f}")

# ---------- PLOTTING ----------

def plot_reliability(p, y, n_bins=10, title="Reliability"):
    bins = np.linspace(0.0, 1.0, n_bins+1)
    centers = 0.5*(bins[:-1]+bins[1:])
    idx = np.digitize(p, bins) - 1
    mean_pred = []
    mean_true = []
    counts = []
    for i in range(n_bins):
        mask = idx == i
        counts.append(mask.sum())
        if mask.sum()==0:
            mean_pred.append(np.nan)
            mean_true.append(np.nan)
        else:
            mean_pred.append(p[mask].mean())
            mean_true.append(y[mask].mean())
    plt.figure(figsize=(5,5))
    plt.plot(mean_pred, mean_true, marker='o', label='Empirical')
    plt.plot([0,1],[0,1],'--', color='gray', label='Perfect')
    plt.xlabel('Mean predicted probability (bin)')
    plt.ylabel('Empirical fraction positive')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

# Per-class reliability plots
for i, col in enumerate((prob0_col, prob1_col, prob2_col)):
    p = df[col].values
    y = (df[true_col].values == i).astype(int)
    plot_reliability(p, y, n_bins=10, title=f"Calibration - class {i}")

eps = 1e-12
labels = df[true_col].values
p_true = probs[np.arange(len(labels)), labels]
nll = -np.mean(np.log(np.clip(p_true, eps, 1.0)))
print(f"NLL (cross-entropy): {nll:.6f}")

# Top-class reliability
plot_reliability(p_max, y_top, n_bins=10, title="Calibration - top predicted class (max prob)")

# Dosage calibration: predicted dosage = P(class1) + 2*P(class2)
dosage_pred = df[prob1_col].values + 2.0*df[prob2_col].values
dosage_true = df[true_col].values.astype(float)
# bin from 0..2
bins = np.linspace(0.0, 2.0, 11)
centers = 0.5*(bins[:-1]+bins[1:])
idx = np.digitize(dosage_pred, bins) - 1
mean_pred = []
mean_true = []
for i in range(len(centers)):
    mask = idx == i
    if mask.sum()==0:
        mean_pred.append(np.nan)
        mean_true.append(np.nan)
    else:
        mean_pred.append(dosage_pred[mask].mean())
        mean_true.append(dosage_true[mask].mean())
plt.figure(figsize=(6,5))
plt.plot(mean_pred, mean_true, marker='o', label='Empirical mean dosage')
plt.plot([0,2],[0,2],'--', color='gray', label='Perfect (y=x)')
plt.xlabel('Mean predicted dosage (bin)')
plt.ylabel('Mean observed dosage')
plt.title('Dosage calibration')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Correlation between actual genotypes and scores expected dosage 

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt

# Assuming your dataframe is already named df
# and has at least these columns:
#   'True_Label' and 'dosage'
df = pd.read_csv('/path/to/data/example_scores_with_maf_dosage.csv')

# Keep only the columns we need
data = df[['True_Label', 'dosage']].copy()
data['True_Label'] = pd.to_numeric(data['True_Label'], errors='coerce')
data['dosage'] = pd.to_numeric(data['dosage'], errors='coerce')
data = data.dropna()

pearson_r, pearson_p = pearsonr(data['True_Label'], data['dosage'])
spearman_rho, spearman_p = spearmanr(data['True_Label'], data['dosage'])

print(f"Rows used: {len(data):,}")
print(f"Pearson r  = {pearson_r:.4f}   (p = {pearson_p:.3e})")
print(f"Spearman ρ = {spearman_rho:.4f}   (p = {spearman_p:.3e})")

labels = sorted(data['True_Label'].unique())
groups = [data.loc[data['True_Label'] == lab, 'dosage'].values for lab in labels]

fig, ax = plt.subplots(figsize=(6, 4))

# Use positions that match the actual labels: 0, 1, 2
ax.boxplot(
    groups,
    positions=labels,
    widths=0.25,
    showfliers=False
)

# Jitter around the true label positions
rng = np.random.default_rng(42)
sample_n = min(5000, len(data))
sample = data.sample(sample_n, random_state=42)

x_jitter = sample['True_Label'].to_numpy() + rng.normal(0, 0.06, size=sample_n)
ax.scatter(x_jitter, sample['dosage'], alpha=0.15, s=10)

# Mean dosage per label
means = data.groupby('True_Label')['dosage'].mean().reindex(labels)
ax.plot(labels, means.values, marker='o', linewidth=2)

ax.set_xticks(labels)
ax.set_xlabel('True Label')
ax.set_ylabel('Brain-genotype scores dosage')
# ax.set_title(f'True_Label vs dosage\nPearson r = {pearson_r:.3f} | Spearman ρ = {spearman_rho:.3f}')
ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
# Reliability plots for each GWAS model (10 tasks within each)


import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss

# ------------------- CONFIG -------------------
input_path = "/path/to/data/example_scores.csv"
group_col  = "GWAS"          # group by model, not Task_ID
task_col   = "Task_ID"       # kept only for reference if needed
true_col   = "True_Label"
prob_cols  = ["Prob_Class_0", "Prob_Class_1", "Prob_Class_2"]
n_bins     = 10
save_plots = True
out_dir    = "./calibration_per_model"
class_names = ["aa", "Aa", "AA"]
# ----------------------------------------------


def safe_name(x):
    """Make a filesystem-safe string."""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))


def ece_bin(p, y, n_bins=10):
    """Binary ECE with equal-width bins, robust to p==1.0."""
    p = np.asarray(p, dtype=float)
    y = np.asarray(y, dtype=float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.digitize(p, bins, right=True) - 1
    idx = np.clip(idx, 0, n_bins - 1)

    ece = 0.0
    for i in range(n_bins):
        mask = (idx == i)
        if mask.sum() == 0:
            continue
        ece += mask.mean() * abs(p[mask].mean() - y[mask].mean())
    return float(ece)


def reliability_curve(p, y, n_bins=10):
    """Return (mean_pred, mean_true, counts) per bin for plotting."""
    p = np.asarray(p, dtype=float)
    y = np.asarray(y, dtype=float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.digitize(p, bins, right=True) - 1
    idx = np.clip(idx, 0, n_bins - 1)

    mean_pred = np.full(n_bins, np.nan, dtype=float)
    mean_true = np.full(n_bins, np.nan, dtype=float)
    counts = np.zeros(n_bins, dtype=int)

    for i in range(n_bins):
        mask = (idx == i)
        counts[i] = int(mask.sum())
        if counts[i] > 0:
            mean_pred[i] = p[mask].mean()
            mean_true[i] = y[mask].mean()

    return mean_pred, mean_true, counts


def multiclass_brier(probs, labels):
    """Multiclass Brier = mean_i sum_c (p_ic - y_ic)^2 (unnormalized by K)."""
    probs = np.asarray(probs, dtype=float)
    labels = np.asarray(labels, dtype=int)
    n = len(labels)
    y_onehot = np.zeros_like(probs)
    y_onehot[np.arange(n), labels] = 1.0
    return float(np.mean(np.sum((probs - y_onehot) ** 2, axis=1)))


def nll_cross_entropy(probs, labels, eps=1e-12):
    """Mean negative log-likelihood using provided probabilities."""
    probs = np.asarray(probs, dtype=float)
    labels = np.asarray(labels, dtype=int)
    p_true = probs[np.arange(len(labels)), labels]
    return float(-np.mean(np.log(np.clip(p_true, eps, 1.0))))


def plot_model_reliability(model_id, probs, labels, n_bins=10, save_path=None):
    """Plot per-class reliability + top-class confidence reliability for one model."""
    fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)

    # Per-class one-vs-rest reliability
    for c in range(3):
        ax = axes[c]
        p = probs[:, c]
        y = (labels == c).astype(int)
        mp, mt, counts = reliability_curve(p, y, n_bins=n_bins)

        m = ~np.isnan(mp) & ~np.isnan(mt)
        ax.plot(mp[m], mt[m], marker="o", label="Empirical")
        ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect")
        # ax.set_title(f"Model {model_id} - class {c}")
        ax.set_title(f"Model {model_id} - {class_names[c]}")
        ax.set_xlabel("Mean predicted p")
        ax.grid(True)

        if c == 0:
            ax.set_ylabel("Empirical probability")

    # Top-class (confidence) reliability
    ax = axes[3]
    pred = probs.argmax(axis=1)
    p_max = probs.max(axis=1)
    y_top = (pred == labels).astype(int)
    mp, mt, counts = reliability_curve(p_max, y_top, n_bins=n_bins)
    m = ~np.isnan(mp) & ~np.isnan(mt)
    ax.plot(mp[m], mt[m], marker="o", label="Empirical")
    ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect")
    ax.set_title(f"Model {model_id} - top-class")
    ax.set_xlabel("Mean confidence (max p)")
    ax.grid(True)

    handles, legend_labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, legend_labels, loc="upper center", ncol=2, frameon=False)
    fig.tight_layout(rect=[0, 0, 1, 0.90])

    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


# ------------------- MAIN -------------------
df = pd.read_csv(input_path)

# Column checks
missing = [c for c in [group_col, true_col, *prob_cols] if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Found: {df.columns.tolist()}")

# Types
df[prob_cols] = df[prob_cols].astype(float)
df[true_col] = df[true_col].astype(int)

if save_plots:
    os.makedirs(out_dir, exist_ok=True)

rows = []

# Group by model (GWAS), not by task
for model_id, g in df.groupby(group_col, sort=True):
    probs = g[prob_cols].to_numpy(float)
    labels = g[true_col].to_numpy(int)
    n = len(g)

    # Sanity checks
    if not np.all(np.isin(labels, [0, 1, 2])):
        raise ValueError(f"Model {model_id}: labels not in {{0,1,2}}. Found {np.unique(labels)}")

    row_sums = probs.sum(axis=1)
    if not np.allclose(row_sums, 1.0, atol=1e-3):
        print(f"Warning Model {model_id}: probs do not sum to 1. "
              f"mean={row_sums.mean():.4f}, min={row_sums.min():.4f}, max={row_sums.max():.4f}")

    # Metrics
    brier_m = multiclass_brier(probs, labels)
    nll = nll_cross_entropy(probs, labels)

    per_class = {}
    for c in range(3):
        p = probs[:, c]
        y = (labels == c).astype(int)
        per_class[f"brier_c{c}"] = float(brier_score_loss(y, p))
        per_class[f"ece_c{c}"] = float(ece_bin(p, y, n_bins=n_bins))

    pred = probs.argmax(axis=1)
    p_max = probs.max(axis=1)
    y_top = (pred == labels).astype(int)
    brier_top = float(brier_score_loss(y_top, p_max))
    ece_top = float(ece_bin(p_max, y_top, n_bins=n_bins))
    acc_top = float(y_top.mean())

    out = {
        "GWAS": model_id,
        "n": n,
        "brier_multi": brier_m,
        "nll": nll,
        "brier_top": brier_top,
        "ece_top": ece_top,
        "acc_top": acc_top,
        **per_class
    }

    # Optional identifiers if present
    if "SNP_rsid" in g.columns:
        out["SNP_rsid"] = g["SNP_rsid"].iloc[0]

    rows.append(out)

    # Plot per model
    if save_plots:
        model_name = safe_name(model_id)
        save_path = os.path.join(out_dir, f"gwas_{model_name}_reliability.png")
        plot_model_reliability(model_id, probs, labels, n_bins=n_bins, save_path=save_path)

metrics_df = pd.DataFrame(rows).sort_values("GWAS")
plot_all_models_reliability(
    df,
    group_col="GWAS",
    prob_cols=prob_cols,
    true_col=true_col,
    n_bins=10,
    save_path="./calibration_per_model/all_models_reliability.png"
)
if save_plots:
    print(f"  - Reliability plots: {out_dir}/gwas_*_reliability.png")

# Quick summary
show_cols = ["GWAS", "n", "brier_multi", "nll", "ece_top", "acc_top"]
print(metrics_df[show_cols].head())

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss

# ---------- CONFIG ----------
input_path = "/path/to/data/example_scores.csv"
group_cols = ["GWAS", "Task_ID"]        
true_col   = "True_Label"
prob_cols  = ["Prob_Class_0", "Prob_Class_1", "Prob_Class_2"]
n_bins     = 10
save_plots = True
out_dir    = "./calibration_per_task_120"
# ---------------------------

def ece_bin(p, y, n_bins=10):
    p = np.asarray(p, dtype=float)
    y = np.asarray(y, dtype=float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.digitize(p, bins, right=True) - 1
    idx = np.clip(idx, 0, n_bins - 1)

    ece = 0.0
    for i in range(n_bins):
        mask = (idx == i)
        if mask.sum() == 0:
            continue
        ece += mask.mean() * abs(p[mask].mean() - y[mask].mean())
    return float(ece)

def multiclass_brier(probs, labels):
    probs = np.asarray(probs, dtype=float)
    labels = np.asarray(labels, dtype=int)
    n = len(labels)
    y_onehot = np.zeros_like(probs)
    y_onehot[np.arange(n), labels] = 1.0
    return float(np.mean(np.sum((probs - y_onehot) ** 2, axis=1)))

def nll_cross_entropy(probs, labels, eps=1e-12):
    probs = np.asarray(probs, dtype=float)
    labels = np.asarray(labels, dtype=int)
    p_true = probs[np.arange(len(labels)), labels]
    return float(-np.mean(np.log(np.clip(p_true, eps, 1.0))))

def plot_task_reliability(task_name, probs, labels, n_bins=10, save_path=None):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)

    def reliability_curve(p, y):
        bins = np.linspace(0.0, 1.0, n_bins + 1)
        idx = np.digitize(p, bins, right=True) - 1
        idx = np.clip(idx, 0, n_bins - 1)

        mean_pred = np.full(n_bins, np.nan)
        mean_true = np.full(n_bins, np.nan)
        for i in range(n_bins):
            m = (idx == i)
            if m.sum():
                mean_pred[i] = p[m].mean()
                mean_true[i] = y[m].mean()
        return mean_pred, mean_true

    # Per-class one-vs-rest
    for c in range(3):
        ax = axes[c]
        p = probs[:, c]
        y = (labels == c).astype(int)
        mp, mt = reliability_curve(p, y)
        m = ~np.isnan(mp) & ~np.isnan(mt)
        ax.plot(mp[m], mt[m], marker="o", label="Empirical")
        ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect")
        ax.set_title(f"{task_name} - class {c}")
        ax.set_xlabel("Mean predicted p")
        ax.grid(True)
        if c == 0:
            ax.set_ylabel("Empirical fraction positive")

    # Top-class confidence calibration
    ax = axes[3]
    pred = probs.argmax(axis=1)
    p_max = probs.max(axis=1)
    y_top = (pred == labels).astype(int)
    mp, mt = reliability_curve(p_max, y_top)
    m = ~np.isnan(mp) & ~np.isnan(mt)
    ax.plot(mp[m], mt[m], marker="o", label="Empirical")
    ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect")
    ax.set_title(f"{task_name} - top-class")
    ax.set_xlabel("Mean confidence (max p)")
    ax.grid(True)

    handles, labs = axes[0].get_legend_handles_labels()
    fig.legend(handles, labs, loc="upper center", ncol=2, frameon=False)
    fig.tight_layout(rect=[0, 0, 1, 0.90])

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

# ---------- LOAD ----------
df = pd.read_csv(input_path)

required = [*group_cols, true_col, *prob_cols]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Found: {df.columns.tolist()}")

df[prob_cols] = df[prob_cols].astype(float)
df[true_col] = df[true_col].astype(int)

# Optional but recommended sanity checks
labels = df[true_col].values
if not np.all(np.isin(labels, [0,1,2])):
    raise ValueError(f"True_Label must be in {{0,1,2}}. Found: {np.unique(labels)}")

row_sums = df[prob_cols].sum(axis=1).values
if not np.allclose(row_sums, 1.0, atol=1e-3):
    print("Warning: probabilities do not sum to 1 (check softmax/logging). "
          f"mean={row_sums.mean():.4f}, min={row_sums.min():.4f}, max={row_sums.max():.4f}")

# Verify each (GWAS, Task_ID) is one SNP (helps catch accidental mixing)
if "SNP_rsid" in df.columns:
    bad = df.groupby(group_cols)["SNP_rsid"].nunique()
    bad = bad[bad > 1]
    if len(bad):
        print("WARNING: Some (GWAS,Task_ID) map to multiple SNP_rsid. "
              "Consider grouping by (GWAS,SNP_rsid) instead. Examples:")
        print(bad.head(10))

# ---------- RUN ----------
if save_plots:
    os.makedirs(out_dir, exist_ok=True)

rows = []
for keys, g in df.groupby(group_cols, sort=True):
    probs = g[prob_cols].to_numpy(float)
    y = g[true_col].to_numpy(int)

    # Metrics
    brier_m = multiclass_brier(probs, y)
    nll = nll_cross_entropy(probs, y)

    per_class = {}
    for c in range(3):
        p = probs[:, c]
        yc = (y == c).astype(int)
        per_class[f"brier_c{c}"] = float(brier_score_loss(yc, p))
        per_class[f"ece_c{c}"] = float(ece_bin(p, yc, n_bins=n_bins))

    pred = probs.argmax(axis=1)
    p_max = probs.max(axis=1)
    y_top = (pred == y).astype(int)
    brier_top = float(brier_score_loss(y_top, p_max))
    ece_top = float(ece_bin(p_max, y_top, n_bins=n_bins))
    acc_top = float(y_top.mean())

    out = {group_cols[0]: keys[0], group_cols[1]: keys[1], "n": len(g),
           "brier_multi": brier_m, "nll": nll,
           "brier_top": brier_top, "ece_top": ece_top, "acc_top": acc_top,
           **per_class}

    if "SNP_rsid" in g.columns: out["SNP_rsid"] = g["SNP_rsid"].iloc[0]
    rows.append(out)

    # Plot name
    task_name = f"GWAS{keys[0]}_Task{keys[1]}" 
    if "SNP_rsid" in g.columns:
        task_name += f"_{g['SNP_rsid'].iloc[0]}"

    if save_plots:
        plot_task_reliability(task_name, probs, y, n_bins=n_bins,
                              save_path=os.path.join(out_dir, f"{task_name}.png"))

metrics_df = pd.DataFrame(rows).sort_values(group_cols)
metrics_df.to_csv(os.path.join(out_dir, "per_task_calibration_metrics.csv"), index=False)
print("Saved per-task metrics to:", os.path.join(out_dir, "per_task_calibration_metrics.csv"))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load metrics
df = pd.read_csv("./calibration_per_task_120/per_task_calibration_metrics.csv")

# Optional: sort GWAS for cleaner plotting
df = df.sort_values("GWAS")

# Set style
sns.set(style="whitegrid")

# Create figure
plt.figure(figsize=(6, 4))

# Boxplot
sns.boxplot(data=df, x='GWAS', y='ece_top')

# Labels and title
plt.title("Top-label ECE per Model (across tasks)")
plt.xlabel("Model")
plt.ylabel("ECE (top-label)")

# Optional: add reference line for interpretation
plt.axhline(df['ece_top'].median(), linestyle='--', color='red')

plt.tight_layout()
plt.savefig('top_class_ece_all_models.png', dpi=300)
plt.show()